In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Загрузка датасетів для проведення аналізу

In [ ]:
# шдях де лежать файли опросу
public = "data/survey_results_public.csv"
schema = "data/survey_results_schema.csv"

In [ ]:
# прочитаємо файл з результатами опросу
df_public = pd.read_csv(public, low_memory=False)

In [ ]:
# визначимо скільки рядків і стовбців має дата-сет
df_public.shape

In [ ]:
# дослідимо сам дата сет з результатами опросу
df_public.sample(10)

In [ ]:
df_public.shape

In [ ]:
# прочитаємо файл з схемою опросу
df_schema = pd.read_csv(schema)

In [ ]:
df_schema.shape

In [ ]:
df_schema.info()

### Завдання 1. Підрахунок загальної кількості респондентів
#### Опис завдання: потрібно визначити загальну кількість респондентів, які взяли участь в опитуванні Stack Overflow

In [ ]:
# призначемо індекс колонці ResponseId для цього спочатку перевіримо чи там не має пустих значень, потім призначимо індекс
# і перевіримо чи там тільки унікальні значення і не має дублікатів
df_public["ResponseId"].isna().sum()

In [ ]:
df_public = df_public.set_index("ResponseId")

In [ ]:
df_public.index.is_unique

In [ ]:
# Оцінемо кількість дублікатів у дата сеті з встановленим індексом
dublicates_exist = df_public.duplicated().any()
if dublicates_exist:
    print("YES")
else:
    print("NO")

In [ ]:
# подивимося на ці дублікати, тому що для pandas NaN == NaN
dublicated_answer = df_public[df_public.duplicated(keep=False)]
dublicated_answer.head(10)

##### Частина респондентів відповіла лише на початкові питання, тоді як більшість запитань залишилися без відповіді.
Через це відповіді таких респондентів виглядають ідентичними за заповненими полями (дублікатами).
<br> Однак ми не можемо не рахувати відповіді цих респондентів або вважати, що вони не приймали участі у опитуванні.

In [ ]:
print(
    f"Кількість респондентів, що відповіли лише на перші запитання: {len(dublicated_answer)}"
)

In [ ]:
# індекси у нас унікальні, без пустих значень
# тоді кількість респондентів - це кількість індексів
print(f"Загальна кількість респондентів: {len(df_public.index)}")

### Висновки завдання 1:

In [ ]:
print(
    f"Загальна кількість респондентів які взяли участь в опитуванні: {len(df_public.index)}"
)

### Завдання 2. Аналіз повноти відповідей респондентів
#### Опис завдання: Визнач, скільки респондентів відповіли на всі запитання опитування.

In [ ]:
#скасовуємо індекс, щоб повернути дата сет в вихідний варіант
df_public = df_public.reset_index()
df_public.shape

In [ ]:
public_columns = set(df_public.columns)
print(f"Кількість колонок у таблиці з відповідями: {len(public_columns)}")

In [ ]:
# перевіряємо чи є в колонці з qid пусті значення
df_schema["qid"].isna().sum()

In [ ]:
# перевіряємо чи є в колонці з qid дублікати
dublicates_schema_exist = df_schema["qid"].duplicated().any()
if dublicates_schema_exist:
    print("YES")
else:
    print("NO")

In [ ]:
qid_unique = set(df_schema["qid"])
print(f"Кількість унікальних qid: {len(qid_unique)}")

In [ ]:
# перевіряємо чи є в колонці з qname пусті значення
df_schema["qname"].isna().sum()

In [ ]:
# перевіряємо чи є в колонці з qname дублікати
dublicates_schema_exist = df_schema["qname"].duplicated().any()
if dublicates_schema_exist:
    print("YES")
else:
    print("NO")

In [ ]:
qname_unique = set(df_schema["qname"])
print(f"Кількість унікальних qname {len(qname_unique)}")

In [ ]:
schema_row = set(df_schema["qname"])
print(f"Кількість рядків у таблиці зі схемою: {len(schema_row)}")

In [ ]:
question_common = public_columns & schema_row
print(
    f"Кількість спільних колонок(df_public) - рядків(df_schema) у файлі схема та у файлі з відповідями: {len(question_common)}"
)

In [ ]:
question_diff = schema_row - question_common
print(f"Кількість питань, що не мають відповідної колонки у файлі з результатами опитування: {len(question_diff)}")
question_diff

In [ ]:
columns_diff = public_columns-question_common
print(f"Кількість колонок у файлі з результатами, що не мають відповідного рядка у файлі зі схемою опитування: {len(columns_diff)}")

#### Обработка питань типу RO, МС, ТЕ у схемі з питаннями (працюємо зі спільними у двох файлах питаннями)

In [ ]:
# робимо копію файла схеми, але залишаємо тільки ті назви qname які є в таблиці з результатами
df_schema_common = df_schema[df_schema["qname"].isin(question_common)].copy()
df_schema_common.shape

In [ ]:
df_schema_common.to_csv(
    r"C:\Users\lena\Projects\Pyton_DA_project\df_schema_common.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# окремо збираємо назви рядків qname які мають одну відповідь і окремо ті які мають мульти-відповіді
df_common_question_answers = (
    df_schema_common.groupby("qid")["qname"].count().reset_index(name="sub_answer_count")
)
print(f"Кількість унікальних qid: {len(df_common_question_answers)}")
print(
    f"Загальна кількість qname (спільних для df_public та df_schema): {df_common_question_answers["sub_answer_count"].sum()}"
)

In [ ]:
df_common_question_answers.sort_values(by="sub_answer_count", ascending=False).head(10)

In [ ]:
# рядкі qid які мають одну відповідь
single_qids = df_common_question_answers.loc[
    df_common_question_answers["sub_answer_count"] == 1, "qid"
].tolist()  # qid питань які мають одну відповідь
single_qids[:7]

In [ ]:
len(single_qids)

In [ ]:
# рядкі qname які мають одну відповідь
single_qnames = list(
    df_schema_common[df_schema_common["qid"].isin(single_qids)]["qname"]
)  # назви qname які мають одну відповідь
single_qnames[:7]

In [ ]:
len(single_qnames)

In [ ]:
# рядкі qname які мають мульті відповіді
multi_qids = df_common_question_answers.loc[
    df_common_question_answers["sub_answer_count"] > 1, "qid"
].tolist()  # qid питань які мають мульті відповіді
multi_qids

In [ ]:
# робимо копію файла df_public з результатами, тому що будемо до нього додавати колонки які є результатом по колонками, які є мульті-відповіддю
df_public_tmp = df_public.copy()

In [ ]:
# для кожного qid яке має мульті відповіді спочатку знаходимо назви qname, які вже є у файлі з результатами і створюємо
# в файлі з результатами нову колоку з назвою питання, в неї записуємо "1" якщо є хоча б одна відповідь у відповідних до цього запитання коллонках
# або NaN якщо жодна з відповідей у відповідних до цього мульти-питання колонках не вибрана
for x in multi_qids:
    q_list = df_schema_common.loc[df_schema_common["qid"] == x, "qname"].tolist()

    df_public_tmp[x] = np.where(
        df_public_tmp[q_list].notna().any(axis=1), 1, np.nan
    )  # 1 якщо хоч в одній колонці до цього qid відповіли

In [ ]:
#перевіряємо чи зробилися колонки з суммарною відповіддю на питання
df_public_tmp.columns

In [ ]:
type(single_qnames)  # перевіряємо тип перед об'єднанням

In [ ]:
type(multi_qids)  # перевіряємо тип перед об'єднанням

In [ ]:
# робимо список колонок для питань типу RO, МС, ТЕ у схемі з питаннями,по яким будемо дивитися чи відповідав респондент
cols_ro_mc_te = single_qnames + multi_qids
print(f"Кількість колонок які включають відповіді на питання: {len(cols_ro_mc_te)}")
cols_ro_mc_te[:10]

#### Обработка питань типу Matrix у схемі з питаннями (працюємо з питаннями які не мають відповідну клонку у файлі з результатами)

In [ ]:
# qname з файлу з схемою, які не є спільними з колонками у файлі з результатами
print(f"Кількість питань, що не мають відповідної колонки у файлі з результатами опитування: {len(question_diff)}")
type(question_diff)

In [ ]:
print(f"Кількість колонок у файлі з результатами, що не мають відповідного рядка у файлі зі схемою опитування: {len(columns_diff)}")
type(columns_diff)

In [ ]:
# знайдемо відповідні кожному qname у схемі, назви колонок у файлі з результатами по тому що частина в назві колонки має бути така як у qname 
# потім для кожного такого набору колонок побудуємо нову колонку де, якщо респондент відповів хоча б в одній з набору колонок  - вважатимемо за відповідь (1)
# пінакще - ні (Nan)
matrix_cols_map = {}  # словник, щоб подивития якому qname відповідають які колонки з файлу з результатами

for x in sorted(question_diff):
    # 1) шукаємо колонки , де назва має в собі х
    cols = [c for c in columns_diff if x in str(c)]

    # (опционально) якщо нічого не знайшли, фіксуємо і продовжуємо
    matrix_cols_map[x] = cols
    if not cols:
        print(f"Для '{x}' не знайдено колонок у columns_diff")
        continue
    # 2) робимо нову колонку x: 1 якщо хоча б одна з cols не NaN, інакше - NaN

    df_public_tmp[x] = np.where(
        df_public_tmp[cols].notna().any(axis=1), 1, np.nan
    )

In [ ]:
matrix_cols_map

In [ ]:
df_public.columns

In [ ]:
df_public_tmp.columns

In [ ]:
type(question_diff)

In [ ]:
type(cols_ro_mc_te)

In [ ]:
# робимо загальний список колонок для питань типу RO, МС, ТЕ та Matrix у схемі з питаннями,по яким будемо дивитися чи відповідав респондент
check_cols = cols_ro_mc_te + list(question_diff)
print(f"Кількість колонок які включають усі відповіді на питання типу RO, MC, NT, Matrix: {len(check_cols)}")
cols_ro_mc_te[:10]


#### Перевіряємо чи на всі питання відповів респондент по списку стовбців з відповідями

In [ ]:
# перевіряємо чи на всі питання відповів респондент по списку стовбців з відповідями, щоб там не було NaN
mask_answered_all = df_public_tmp[check_cols].notna().all(axis=1)

In [ ]:
number_answered_all = int(mask_answered_all.sum())
print(
    f"Кількість респондентів, які відповіли на усі запитання опросу: {number_answered_all}"
)

In [ ]:
# Вивидемо дані по цьому респонденту
df_public_tmp.loc[mask_answered_all, :]

In [ ]:
df_resp_all_answer = df_public_tmp.loc[mask_answered_all, :]

df_resp_all_answer.to_csv(
    r"C:\Users\lena\Projects\Pyton_DA_project\respondents_all_answers.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Вивидемо дані по цьому респонденту
id_all_guestion_answer = int(df_public_tmp.loc[mask_answered_all, ["ResponseId"]].iloc[0, 0])
print(f"Id респондента, який відповів на усі запитання: {id_all_guestion_answer}")

In [ ]:
# - побачимо топ самих заповнених відповідей
df_filled_per_person = df_public_tmp[check_cols].notna().sum(axis=1)
df_filled_per_person.sort_values(ascending=False).head(10)

### Висновки завдання 2:

In [ ]:
print(
    f"Кількість респондентів, які відповіли на усі запитання опросу: {number_answered_all}"
)
print(f"Id респондента, який відповів на усі запитання: {id_all_guestion_answer}")

### Завдання 3. Статистичний аналіз досвіду респондентів
#### Опис завдання: Обчисли міри центральної тенденції для поля WorkExp (досвід роботи респондентів). 
Тобі потрібно знайти:
* Середнє значення (mean)
* Медіану (median)
* Моду (mode)

In [ ]:
# Спочатку проаналізуємо данні у дата сеті по колонці WorkExp
index_workexp = df_public.columns.get_loc("WorkExp")

In [ ]:
df_public.iloc[:, : (index_workexp + 1)].sample(10)

In [ ]:
# перевіряємо чи є в колонці з WorkExp пусті значення
null_workexp = int(df_public["WorkExp"].isna().sum())
print(f"Кількість пустих даних в колонці WorkExp: {null_workexp}")

In [ ]:
all_workexp = len(df_public["WorkExp"])
czastka = round(null_workexp / all_workexp * 100, 2)
print(
    f"Частка респондентів, які не відповіли на пиання повязане з досвідом роботи складає:{czastka} %"
)

In [ ]:
# залишимо тільки необхідні для аналізу колонки
df_public_workexp = df_public[["ResponseId", "Age", "WorkExp"]]
df_public_workexp.head()

In [ ]:
# подивимося який розподіл значень у колонці WorkExp
df_public_workexp["WorkExp"].hist(color="#BB2255")

plt.title("Розподіл років досвіду серед респондентів опросу")
plt.xlabel("роки досвіду")
plt.ylabel("Частота")
plt.show()

In [ ]:
# по гистограмі бачимо, що у відповідях є недостовірні дані - 60 і навіть 100 років досвіду
grouped = df_public_workexp.groupby("Age")["WorkExp"].agg(
    ["count", "min", "median", "mean", "max"]
)
grouped.head(10)

In [ ]:
workexp_counts = (
    df_public_workexp["WorkExp"]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)
workexp_counts.head(10)

In [ ]:
workexp100 = len(df_public_workexp[df_public_workexp["WorkExp"] == 100])
print(
    f"Кількість респондентів, що вказали свій досвід роботи як 100 років: {workexp100}"
)

In [ ]:
# подивимося як по категоріях в Age відповіли респонденти про свій вік досвіду- classik pivot table
pivot_age_workexp = pd.pivot_table(
    df_public_workexp,
    index="Age",
    columns="WorkExp",
    values="ResponseId",
    aggfunc="count",
    fill_value=0,
    dropna=False,
)

pivot_age_workexp

#### Висновки з проведеного аналізу даних відображаючих досвід респондентів:
* Значення показника досвіду роботи (WorkExp) містить 6298 пропущених значень, що становить 12,8% респондентів.
Це означає, що частина учасників не вказала свій досвід роботи.
* Крім того значення показника досвіду роботи (WorkExp) містять екстремальні значення (наприклад, 99 або 100 років),
що не можуть бути інтерпретовані буквально.
* Тому для аналізу доречно використосвувати медіану, 
яка є більш стійкою до викидів і краще відображає типовий рівень досвіду в кожній віковій групі.

In [ ]:
# розрахуємо потрібні у завданні міри центральноїтенденціі
workexp = df_public["WorkExp"]
mean_val = workexp.mean()  # NaN ігнорується
median_val = workexp.median()  # NaN ігнорується але може буть кілька мод
mode_val = workexp.mode()  # NaN ігнорується але може буть кілька мод

print(type(mode_val))  # перевіримо тип mode_val якщо Series, то мод може бути декілька
print(mode_val.size)

In [ ]:
print("Середнє:          ", round(mean_val, 2))
print("Мода:             ", mode_val.iloc[0])
print("Медіана:          ", median_val)

Для обчислення мір центральної тенденції використовуються лише ті спостереження, 
<br>де WorkExp заповнений, оскільки NaN означає відсутність інформації, а не нульовий досвід.
<br><br>Заміна NaN на 0 є некоректною, оскільки відсутня відповідь не означає відсутність досвіду/

### Висновки завдання 3:

In [ ]:
print("Mіри центральної тенденції для поля WorkExp (досвід роботи респондентів):")
print("Середнє:          ", round(mean_val, 2))
print("Мода:             ", mode_val.iloc[0])
print("Медіана:          ", median_val)

### Завдання 4. Аналіз віддаленої роботи
#### Опис завдання: Визнач кількість респондентів, які працюють віддалено (remote). 
Тобі потрібно знайти відповідне поле в датасеті та підрахувати кількість працівників, які обрали цей формат роботи.

In [ ]:
# Спочатку проаналізуємо данні у дата сеті по колонці RemoteWork
# залишимо тільки необхідні для аналізу колонки
df_public_remotework = df_public[["ResponseId", "RemoteWork"]]
df_public_remotework.head()

In [ ]:
df_public_remotework.describe(include="all")

In [ ]:
df_remotework_counts = (
    df_public_remotework["RemoteWork"].value_counts(dropna=False).reset_index()
)
df_remotework_counts.head(10)

In [ ]:
# розрахуємо частку, яку займає кожна категорія у загальній кількості відповідей респондентів
total = df_remotework_counts["count"].sum()

df_remotework_counts["percent"] = (df_remotework_counts["count"] / total * 100).round()
df_remotework_counts

In [ ]:
# подивимося які варіанти відповіді як опції були запропоновані респондентам
options_answer = set(df_public_remotework["RemoteWork"])
options_answer

In [ ]:
# для відповіді на питання розділемо усі відповіді на чотири категорії:
# 1) ті хто зазнчив що він працює виключно віддалено - 'Remote'Hybrid (some in-pe
# 2) ті, хто може працювати також віддалено - 'rson, leans heavy to flexibility)',
#                                            'Hybrid (some remote, leans heavy to in-person)',
#                                            'Your choice (very flexible, you can come in when you want or just as needed)'
# 3) ті хто зазнчив що він працює виключно в офісі - 'In-person'
# 4 ті хто не надав відповіді - 'NaN'
# Для відповіді виведемо кількість респондентів по категорії 1 та 2

conditionally_remote = [
    "Remote",
    "Hybrid (some remote, leans heavy to in-person)",
    "Hybrid (some in-person, leans heavy to flexibility)",
    "Your choice (very flexible, you can come in when you want or just as needed)",
]


def categorize_remote(x):
    if x == "Remote":
        return "Віддалена або частково віддалена робота"
    elif x == "In-person":
        return "Робота в офісі"
    elif x in conditionally_remote:
        return "Віддалена або частково віддалена робота"
    else:
        return "Не дали відповіді"


df_remotework_counts["categorie"] = df_remotework_counts["RemoteWork"].apply(
    categorize_remote
)
df_grouped = df_remotework_counts.groupby("categorie", as_index=False)[
    ["count", "percent"]
].sum()
df_grouped.sort_values(by="categorie")


In [ ]:
mask = df_grouped["categorie"].isin(
    ["Віддалена або частково віддалена робота"]
)

combined_row = pd.DataFrame(
    [
        {
            "categorie": "Віддалена або частково віддалена робота",
            "count": df_grouped.loc[mask, "count"].sum(),
            "percent": df_grouped.loc[mask, "percent"].sum(),
        }
    ]
)

combined_row

In [ ]:
remote_only = (df_public_remotework["RemoteWork"] == "Remote").sum()
print(
    f"з них - кількість респондентів, які вказали що працюють виключно віддалено: {int(remote_only)}"
)

In [ ]:
remote_all = df_public_remotework["RemoteWork"].isin(conditionally_remote).sum()
print(
    f"Кількість респондентів, які мають можливість праювати віддалено (повністю або частково): {int(remote_all)}"
)

### Висновки завдання 4:

In [ ]:
conditionally_remote = [
    "Remote",
    "Hybrid (some remote, leans heavy to in-person)",
    "Hybrid (some in-person, leans heavy to flexibility)",
    "Your choice (very flexible, you can come in when you want or just as needed)",
]
remote_all = df_public_remotework["RemoteWork"].isin(conditionally_remote).sum()
remote_only = (df_public_remotework["RemoteWork"] == "Remote").sum()
print(
    f"Кількість респондентів, які мають можливість працювати віддалено (повністю або частково): {int(remote_all)}"
)
print(
    f"з них - кількість респондентів, які вказали що працюють виключно віддалено: {int(remote_only)}"
)

### Завдання 5. Визначення популярності Python
#### Опис завдання: Обчисли відсоток респондентів, які програмують на Python. 
Зверни увагу, що поле з мовами програмування може містити множинні значення, тому потрібно перевірити наявність Python в цьому полі.

In [ ]:
col_names = [
    "LanguageChoice",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
    "LanguagesHaveEntry",
    "LanguagesWantEntry",
]
df_public[col_names].sample(10)

При оцінці існуючих в опросі питань які стосуються мови программування, було виявлене наступне -
в опитуванні є питання про мову програмування, відповідь на які заноситься у наступні колонки:
1) LanguageHaveWorkedWith - Програмували протягом минулого року
2) LanguageWantToWorkWith - Хочуть програмувати 
3) LanguageAdmired - Вважають трендовим
4) LanguagesHaveEntry - Треба внести назву мови программування, якщо програмували протягом минулого року, але не вказано серед мов программування
5) LanguagesWantEntry - Треба внести назву мови программування, якщо хочуть програмувати, але не вказано серед мов программування
Висновок:
У межах даного завдання під респондентами, які програмують на Python, будемо розуміти тих, хто використовував Python протягом останнього року, відповідно до поля LanguageHaveWorkedWith. Тому що це підтверджений досвід (а не подобання чи плани)

In [ ]:
# Відберемо тільки тих респондентів, у яких в полі LanguageHaveWorkedWith серед перерахованих є Python
mask = df_public["LanguageHaveWorkedWith"].str.contains(
    "Python", na=False
)  # перевіримо чи є строка Python в полі, якщо там NaN то рахуємо як False
df_python_programiers = df_public[mask]
# для перевірки
for val in df_public.loc[mask, "LanguageHaveWorkedWith"].sample(10):
    print(val)

In [ ]:
df_python_programiers[["ResponseId", "LanguageHaveWorkedWith"]].describe(include="all")

In [ ]:
# простий спосіб дати відповідь на питання завдання 5
total_responce = len(df_public["LanguageHaveWorkedWith"])
python_use = int(mask.sum())
percent_python = python_use / total_responce * 100
df_python_used = pd.DataFrame(
    [
        {
            "Категорія": "Bідсоток респондентів, які програмують на Python",
            "Кількість": python_use,
            "Процент %": round(percent_python, 2),
        }
    ]
)

df_python_used

In [ ]:
# інший спосіб дати відповідь на питання завдання 5
df_public["python_use"] = np.where(
    mask, 1, 0
)  # робимо новий стовбець - 1 якщо програмував і 0 якщо ні
df_python_use = df_public["python_use"].value_counts().reset_index()
df_python_use["percent_python_use"] = round(
    ((df_python_use["count"] / df_python_use["count"].sum()) * 100), 2
)
df_python_use

### Висновки завдання 5:

In [ ]:
total_responce = len(df_public["LanguageHaveWorkedWith"])
python_use = int(mask.sum())
percent_python = python_use / total_responce * 100
df_python_used = pd.DataFrame(
    [
        {
            "Категорія": "Bідсоток респондентів, які програмують на Python",
            "Кількість": python_use,
            "Процент %": round(percent_python, 2),
        }
    ]
)

df_python_used

### Завдання 6. Аналіз шляхів навчання програмуванню
#### Опис завдання: Визнач, скільки респондентів навчалося програмувати за допомогою онлайн курсів. 
Знайди відповідне поле в датасеті, яке описує способи навчання, та підрахуй тих, хто обрав онлайн курси. 

In [ ]:
# пошукаємо відповідні колонки у таблиці з результатами
pd.set_option(
    "display.max_colwidth", None
)  # встановимо максимальну довжину колонки, щоб бачити усі відповіді
col_names = [
    "LearnCodeChoose",
    "LearnCode",
    "LearnCodeAI",
    "LanguageAdmired",
    "AILearnHow",
    "YearsCode",
]
df_public[col_names].sample(5)

In [ ]:
# відберемо тільки потрібні нам колонки
col_names = ["ResponseId", "LearnCode"]
df_learn_code = df_public[col_names]
df_learn_code.sample(5)

In [ ]:
# спробуємо оцінити скільки унікальних вариантів відповіді існує на це запитання
df_learn_code_counts = (
    df_learn_code["LearnCode"].value_counts().sort_index(ascending=False).reset_index()
)
df_learn_code_counts.sample(5)

In [ ]:
# в описі опитувальника на цьому питанні представлен чекбокс с можливістю обирання кілька відповідей - тобто щоб знайти тих,
# хто обрав Online Courses or Certification (includes all media types) треба перевірити входження в множину
# Відберемо тільки тих респондентів, у яких в полі LearnCode серед перерахованих є Online Courses or Certification (includes all media types)
mask_learn = df_public["LearnCode"].str.contains(
    "Online Courses or Certification (includes all media types)", regex=False, na=False
)
df_online_courses_learn = df_public[mask_learn]

In [ ]:
print(
    f"Кількість респондентів, які навчалися програмувати за допомогою онлайн курсів: {len(df_online_courses_learn)}"
)

In [ ]:
pd.reset_option(
    "display.max_colwidth"
)  # знімаємо установку на максимальну довжину колонки

### Висновки завдання 6:

In [ ]:
print(
    f"Кількість респондентів, які навчалися програмувати за допомогою онлайн курсів: {len(df_online_courses_learn)}"
)

### Завдання 7. Географічний аналіз компенсації Python-розробників
#### Опис завдання: Серед респондентів, що програмують на Python, проведи аналіз компенсації (ConvertedCompYearly) в розрізі країн.
Для кожної країни обчисли:
* Середню суму компенсації
* Медіанну суму компенсації

In [ ]:
# Відберемо тільки тих респондентів, у яких в полі LanguageHaveWorkedWith серед перерахованих є Python
mask = df_public["LanguageHaveWorkedWith"].str.contains(
    "Python", na=False
)  # перевіримо чи є строка Python в полі, якщо там NaN то рахуємо як False
df_python_programiers = df_public[mask]
col_name = [
    "ResponseId",
    "ConvertedCompYearly",
    "Country",
    "Currency",
    "CompTotal",
    "LanguageHaveWorkedWith",
]
df_python_programiers_comp = df_python_programiers[col_name]
df_python_programiers_comp.sample(5)

In [ ]:
# приберемо з таблиці дані респондентів, які не вказали свою компенсацію
df_python_comp = df_python_programiers_comp[
    df_python_programiers_comp["ConvertedCompYearly"].notna()
]
df_python_group = (
    df_python_comp.groupby("Country")["ConvertedCompYearly"]
    .agg(["count", "sum", "median", "mean"])
    .round(2)
)
df_python_group.sort_values(by="mean", ascending=False).reset_index()

In [ ]:
df_python_group[["count", "median", "mean"]].sort_values(
    by="mean", ascending=False
).head(10)

### Висновки завдання 7:

При аналізі компенсації Python-розробників з вибірки були виключені респонденти, які не вказали річну компенсацію (ConvertedCompYearly = NaN), оскільки за відсутності цього значення неможливо коректно обчислити середню та медіанну компенсацію. Аналіз проведено лише для країн, у яких наявні валідні дані щодо компенсації.

In [ ]:
df_python_group[["count", "median", "mean"]].sort_values(
    by="mean", ascending=False
).head(10)

<br>Аналіз показує, що у більшості країн середнє значення компенсації перевищує медіанне, що свідчить про наявність високих доходів у окремих респондентів та асиметричний розподіл доходів. У країнах з великим розривом між середнім та медіанним значенням (наприклад, Viet Nam та Nigeria) середня компенсація є нерепрезентативною для типової ситуації, оскільки на неї суттєво впливають поодинокі дуже високі значення. У таких випадках медіана є більш надійним показником типової компенсації Python-розробника.

<br>Для країн із більш збалансованим розподілом доходів (наприклад, United States або Switzerland) різниця між середнім та медіанним значенням є меншою, що дозволяє використовувати обидва показники для оцінки рівня компенсації.

<br>У країнах де медіанне значенні дорівнює середньому (наприклад Оман, Андора) це пов'язане з дуже малою кількістю респондентів, які вказали рівень своєї компенсації.


### Завдання 8. Аналіз освіти найбільш оплачуваних спеціалістів
#### Опис завдання: Визнач рівні освіти 5 респондентів з найбільшою компенсацією. 
Тобі потрібно:
* Відсортувати респондентів за рівнем компенсації
* Вибрати топ-5
* Показати їх рівні освіти

In [ ]:
col_name = ["ResponseId", "ConvertedCompYearly", "EdLevel"]
df_top5_edlevel = (
    df_public[col_name].sort_values(by="ConvertedCompYearly", ascending=False).head(5)
)
df_top5_edlevel

In [ ]:
ed_level_top = set(df_top5_edlevel["EdLevel"])
ed_level_top

In [ ]:
print("Рівні освіти 5 респондентів з найбільшою компенсацією:")
for level in ed_level_top:
    print(f"- {level}")

### Висновки завдання 8:

In [ ]:
print("Рівні освіти 5 респондентів з найбільшою компенсацією:")
for level in ed_level_top:
    print(f"- {level}")
df_top5_edlevel

### *Завдання 9. Аналіз популярності Python по віковим категоріям 
#### Опис завдання: Для кожної вікової категорії визнач відсоток респондентів, які програмують на Python. 
Це дасть уявлення про те, в яких вікових групах Python є найпопулярнішим.

In [ ]:
# перевіримо чи залишилася колонка з маскою фільтруючою тих респондентів, які програмують на Python яку ми робили у завданні 5
df_public["python_use"]

In [ ]:
# Обираємо тільки потрібні нам колонки з усього дата-сету з рузультатами опитування
col_name = ["ResponseId", "Age", "python_use"]
df_public[col_name].head(5)

In [ ]:
# подивимося як по категоріях в Age відповіли респонденти про використання ними Python - classik pivot table
pivot_age_python = pd.pivot_table(
    df_public,
    index="Age",
    columns="python_use",
    values="ResponseId",
    aggfunc="count",
    fill_value=0,
)
pivot_age_python

In [ ]:
# додаємо дві розрахункові колонки з загальною кількістю по кожній віковій группі та процент по кожній групі тих хто використовує Python
pivot_age_python["total"] = pivot_age_python[0] + pivot_age_python[1]
pivot_age_python["Percent_used_python"] = (
    pivot_age_python[1] / pivot_age_python["total"] * 100
).round(2)
pivot_age_python.sort_values(by="Percent_used_python", ascending=False)

In [ ]:
#підготуємо красивий формат для відповіді на завдання
max_val_percent = pivot_age_python["Percent_used_python"].max()
df_max_val_percent = pivot_age_python[
    pivot_age_python["Percent_used_python"] == max_val_percent
]
df_max_val_percent

In [ ]:
#підготуємо красивий формат для відповіді на завдання
name_max_val_percent = str(df_max_val_percent.index[0])
name_max_val_percent

### Висновки завдання 9:

При розрахунку відсотка респондентів, які програмують на Python, пропущені значення не виключались з аналізу, оскільки вони означають, що респондент не вказав використання Python.
Таким чином, відсоток розраховано від загальної кількості респондентів у кожній віковій групі, що дозволяє коректно оцінити популярність Python.

In [ ]:
print(f"В віковій групі '{name_max_val_percent}' -  Python є найпопулярнішим")
pivot_age_python = pivot_age_python.rename(
    columns={"Percent_used_python": "Respondents used Python,%"}
)
pivot_age_python
pivot_age_python[["Respondents used Python,%"]].sort_values(by="Respondents used Python,%", ascending=False)

### *Завдання 10. Аналіз індустрій серед високооплачуваних віддалених працівників
#### Опис завдання: Знайди респондентів, які:
* Знаходяться у 75 перцентилі за компенсацією (тобто заробляють більше, ніж 75% респондентів)
* Працюють віддалено
* Для цієї групи визнач найрозповсюдженіші індустрії.

In [ ]:
col_name = ["ResponseId", "ConvertedCompYearly", "RemoteWork", "Industry"]
df_industries_comp = df_public[col_name]
df_industries_comp.sample(5)

In [ ]:
len(df_industries_comp)

In [ ]:
#Розраховуємо границю 75 перцентилю за компенсацією (тобто знаходимо границю для тих, хто заробляє більше, ніж 75% респондентів)
p75 = float(df_industries_comp["ConvertedCompYearly"].quantile(0.75))
mask_p75 = df_industries_comp["ConvertedCompYearly"] >= p75 # маска яку засттосуємо, щоб залишити тільки тих, хто заробляє більше

In [ ]:
# Знайдемо респоднетів, які знаходяться у 75 перцентилі за компенсацією (тобто заробляють більше, ніж 75% респондентів)
df_industries_p75 = df_industries_comp[mask_p75]
len(df_industries_p75)  

In [ ]:
conditionally_remote = [
    "Remote",
    "Hybrid (some remote, leans heavy to in-person)",
    "Hybrid (some in-person, leans heavy to flexibility)",
    "Your choice (very flexible, you can come in when you want or just as needed)",
]
mask_remote = df_industries_p75["RemoteWork"].isin(conditionally_remote) # маска яку засттосуємо, щоб залишити тільки тих, хто працює віддалено (повністю або частково)

In [ ]:
# Знайдемо респоднетів, які працюють віддалено(повністю або частково)
df_industries_p75_remote = df_industries_p75[mask_remote]
len(df_industries_p75_remote)

In [ ]:
# перевіримо чи є в колонці Industry пусті значення (в колонках ConvertedCompYearly та RemoteWork їх не може бути завдяки попередньому відбору)
df_industries_p75_remote["Industry"].isna().sum()

In [ ]:
# перевіримо чи є в колонці RemoteWork пусті значення 
df_industries_p75_remote["RemoteWork"].isna().sum()

In [ ]:
# перевіримо чи є в колонці ConvertedCompYearly значення (
df_industries_p75_remote["ConvertedCompYearly"].isna().sum()

In [ ]:
#Прибираємо дані де у колонці Industry є NaN тобто немає відповіді
df_industries_p75_remoter = df_industries_p75_remote[(df_industries_p75_remote["Industry"].notna())]
len(df_industries_p75_remoter)

In [ ]:
#згпупуємо за індустріями
df_industries_group = (
    df_industries_p75_remote.groupby("Industry")[["ResponseId"]]
    .count()
    .sort_values(by="ResponseId", ascending=False)
)
df_industries_group.head(10)

In [ ]:
#розрахуємо процент
total_p75_remote_industries = int(df_industries_group["ResponseId"].sum())
print(f"total_p75_remote_industries= {total_p75_remote_industries}")


In [ ]:
df_industries_group["Percentage of respondents"] = round((df_industries_group["ResponseId"] / total_p75_remote_industries *100),2)
df_industries_group

In [ ]:
#підготуємо красивий формат для відповіді на завдання
name_p75_industries = df_industries_group.index[0:5].to_list()
name_p75_industries

### Висновки завдання 10:

In [ ]:
print("Cеред високооплачуваних віддалених працівників найрозповсюдженіші індустрії:")
for ind in name_p75_industries:
    print(f"- {ind}")

#df_industries_group = df_industries_group.rename(
#    columns={"ResponseId": "Number of survey respondents"}
#)
df_industries_group